# Faces and Total Pages by Decade

This notebook combines the METS download log with the deduplicated face-detection CSV to summarize page volume and face presence by decade.

Execution convention:

Run this notebook from its own directory, `code/scripts`. The repository's VS Code setting `jupyter.notebookFileRoot = ${fileDirname}` makes that the intended execution directory.

Inputs:

- `../../data/metadata/economist_mets_download_log.json`
- `../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv`

Outputs:

- `../../data/processed/faces_pages_by_decade.csv`
- `../../code/output/figures/faces_pages_by_decade.svg`
- `../../code/output/figures/issues_with_detected_faces.svg`

Counting rules:

- `total_pages` is the sum of `default_file_count` from the METS download log.
- `detected_faces` is the count of deduplicated face detections.
- `pages_with_faces` counts unique actual page numbers with at least one retained face detection. The page-number list is parsed from the filename stem, so a source scan such as `0045,0046` contributes two pages when both pages are part of the scan.
- The summary is restricted to decades from 1900 onward.
- Because the local METS log currently runs through 2007, face detections from later years are excluded from the decade summary so all ratios use a valid total-page denominator.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import seaborn as sns

pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 140
sns.set_theme(style="whitegrid")

ANALYSIS_START_DECADE = 1900

download_log_json = Path("../../data/metadata/economist_mets_download_log.json")
deduplicated_faces_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
summary_csv = Path("../../data/processed/faces_pages_by_decade.csv")
summary_figure_svg = Path("../../code/output/figures/faces_pages_by_decade.svg")
issue_incidence_figure_svg = Path("../../code/output/figures/issues_with_detected_faces.svg")

assert download_log_json.exists(), f"Missing METS download log: {download_log_json}"
assert deduplicated_faces_csv.exists(), f"Missing deduplicated face CSV: {deduplicated_faces_csv}"
summary_csv.parent.mkdir(parents=True, exist_ok=True)
summary_figure_svg.parent.mkdir(parents=True, exist_ok=True)
issue_incidence_figure_svg.parent.mkdir(parents=True, exist_ok=True)

expected_face_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

print(f"METS log:      {download_log_json}")
print(f"Face CSV:      {deduplicated_faces_csv}")
print(f"Summary CSV:   {summary_csv}")
print(f"Figure output: {summary_figure_svg}")
print(f"Incidence fig: {issue_incidence_figure_svg}")

## Load and Validate METS Page Totals

The METS download log supplies the decade denominator. This cell validates the core schema, parses years from `issue_id`, restricts the data to decades from 1900 onward, and aggregates total pages by decade.

In [ ]:
download_log = pd.DataFrame(json.loads(download_log_json.read_text(encoding="utf-8")))

required_log_columns = {"issue_id", "default_file_count", "status", "path"}
missing_log_columns = required_log_columns.difference(download_log.columns)
assert not missing_log_columns, {"missing_log_columns": sorted(missing_log_columns)}
assert len(download_log) > 0, "The METS download log is empty."
assert download_log["issue_id"].is_unique, "The METS download log contains duplicate issue_id values."
assert download_log["default_file_count"].notna().all(), "Some METS rows are missing default_file_count."

issue_parts = download_log["issue_id"].str.extract(r"^ECON-(?P<year>\d{4})-(?P<monthday>\d{4})$")
assert issue_parts.notna().all().all(), "Some METS issue_id values could not be parsed."

download_log = download_log.copy()
download_log["year"] = pd.to_numeric(issue_parts["year"], errors="raise").astype("int64")
download_log["decade"] = (download_log["year"] // 10) * 10
download_log["issue_id_short"] = download_log["issue_id"].str.removeprefix("ECON-")
download_log["issue_date"] = pd.to_datetime(download_log["issue_id_short"], format="%Y-%m%d", errors="raise")
download_log["default_file_count"] = pd.to_numeric(download_log["default_file_count"], errors="raise").astype("int64")
download_log = download_log.loc[download_log["decade"] >= ANALYSIS_START_DECADE].copy()

assert (download_log["default_file_count"] > 0).all(), "Some METS rows have non-positive page counts."
allowed_statuses = {"downloaded", "skipped_existing"}
observed_statuses = set(download_log["status"].dropna())
assert observed_statuses, "The filtered METS log has no status values."
assert observed_statuses.issubset(allowed_statuses), {
    "unexpected_statuses": sorted(observed_statuses.difference(allowed_statuses))
}

decade_page_totals = (
    download_log.groupby("decade", as_index=False)
    .agg(
        issues_in_log=("issue_id", "size"),
        total_pages=("default_file_count", "sum"),
        first_issue_date=("issue_date", "min"),
        last_issue_date=("issue_date", "max"),
    )
    .sort_values("decade")
    .reset_index(drop=True)
)

assert decade_page_totals["decade"].is_monotonic_increasing
assert decade_page_totals["total_pages"].gt(0).all()

print(f"Loaded {len(download_log):,} METS issue records across {len(decade_page_totals):,} decades.")
decade_page_totals.head()

## Load and Validate the Deduplicated Face Dataset

The face CSV contributes two decade-level numerators:

- `detected_faces`: one row per retained face detection;
- `pages_with_faces`: unique page numbers after exploding the filename page list.

The notebook keeps only face rows whose issue IDs are covered by the local METS log and whose decade is 1900 or later so the decade ratios use matching coverage.

In [ ]:
faces = pd.read_csv(deduplicated_faces_csv, dtype={"Filename": "string"})

assert list(faces.columns) == expected_face_columns, {
    "expected": expected_face_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The deduplicated face CSV is empty."

filename_parts = faces["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)
parse_failures = int(filename_parts["issue_id"].isna().sum())
assert parse_failures == 0, f"Could not parse {parse_failures:,} face filenames."

faces = faces.copy()
faces["issue_id"] = filename_parts["issue_id"]
faces["source_pages"] = filename_parts["source_pages"]
faces["year"] = faces["issue_id"].str.slice(0, 4).astype("int64")
faces["decade"] = (faces["year"] // 10) * 10

faces = faces.loc[faces["decade"] >= ANALYSIS_START_DECADE].copy()

available_issue_ids = set(download_log["issue_id_short"])
covered_mask = faces["issue_id"].isin(available_issue_ids)
excluded_face_rows = int((~covered_mask).sum())
faces = faces.loc[covered_mask].copy()

assert len(faces) > 0, "No face rows remain after restricting to issues covered by the METS log."
assert faces["issue_id"].isin(available_issue_ids).all()
assert faces["year"].min() >= int(download_log["year"].min())
assert faces["year"].max() <= int(download_log["year"].max())

face_pages = faces[["issue_id", "decade", "source_pages"]].drop_duplicates().copy()
face_pages["page_number"] = face_pages["source_pages"].str.split(",")
face_pages = face_pages.explode("page_number", ignore_index=True)
face_pages["page_number"] = pd.to_numeric(face_pages["page_number"], errors="raise").astype("int64")
face_pages = face_pages[["issue_id", "decade", "page_number"]].drop_duplicates().reset_index(drop=True)

assert face_pages["page_number"].gt(0).all()

detected_faces_by_decade = (
    faces.groupby("decade", as_index=False)
    .agg(
        issues_with_faces=("issue_id", "nunique"),
        detected_faces=("Filename", "size"),
    )
    .sort_values("decade")
    .reset_index(drop=True)
)

pages_with_faces_by_decade = (
    face_pages.groupby("decade", as_index=False)
    .agg(pages_with_faces=("page_number", "size"))
    .sort_values("decade")
    .reset_index(drop=True)
)

print(f"Loaded {len(faces):,} retained face detections across {faces['issue_id'].nunique():,} issues covered by the METS log.")
print(f"Excluded {excluded_face_rows:,} face rows from issues outside the METS log coverage.")
detected_faces_by_decade.head()

## Combine the Decade Totals

This step merges the page denominator with the face-based numerators and derives relative measures for plotting and export. It also creates a display label that marks the partial 2000s coverage explicitly with an asterisk.

In [ ]:
decade_summary = (
    decade_page_totals
    .merge(detected_faces_by_decade, on="decade", how="left")
    .merge(pages_with_faces_by_decade, on="decade", how="left")
    .sort_values("decade")
    .reset_index(drop=True)
)

for column in ["issues_with_faces", "detected_faces", "pages_with_faces"]:
    decade_summary[column] = decade_summary[column].fillna(0).astype("int64")

decade_summary["pages_with_faces_share"] = decade_summary["pages_with_faces"] / decade_summary["total_pages"]
decade_summary["pages_with_faces_share_percent"] = decade_summary["pages_with_faces_share"] * 100
decade_summary["pages_with_faces_per_100_pages"] = decade_summary["pages_with_faces"] / decade_summary["total_pages"] * 100
decade_summary["detected_faces_per_100_pages"] = decade_summary["detected_faces"] / decade_summary["total_pages"] * 100
max_decade = int(decade_summary["decade"].max())
max_year = int(pd.to_datetime(decade_summary.loc[decade_summary["decade"] == max_decade, "last_issue_date"]).dt.year.max())
decade_summary["period_label"] = decade_summary["decade"].astype(str) + "s"
decade_summary.loc[decade_summary["decade"] == max_decade, "period_label"] = f"{max_decade}s*"

assert (decade_summary["pages_with_faces"] <= decade_summary["total_pages"]).all()
assert (decade_summary["detected_faces"] >= decade_summary["pages_with_faces"]).all()
assert decade_summary["decade"].is_unique

display_columns = [
    "decade",
    "issues_in_log",
    "issues_with_faces",
    "total_pages",
    "detected_faces",
    "pages_with_faces",
    "pages_with_faces_per_100_pages",
    "detected_faces_per_100_pages",
    "period_label",
]

decade_summary[display_columns]

## Write Outputs and Plot the Decade Summary

The figure uses two panels:

- left: total pages and detected faces by decade;
- right: pages with faces together with two rate measures, pages with faces per 100 pages and detected faces per 100 pages.

The CSV preserves the merged decade summary for reuse elsewhere in the thesis workflow.

In [ ]:
decade_summary.to_csv(summary_csv, index=False)

plot_summary = decade_summary.copy()
plot_summary["decade_label"] = plot_summary["period_label"]
x = np.arange(len(plot_summary))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))
fig.subplots_adjust(bottom=0.08, wspace=0.46)

ax_left = axes[0]
left_bar = ax_left.bar(x, plot_summary["total_pages"], color="#4c78a8", alpha=0.9, label="Total pages")
ax_left.set_title("Total pages and detected faces by decade")
ax_left.set_ylabel("Total pages")
ax_left.set_xticks(x, plot_summary["decade_label"], rotation=45, ha="right")
left_total_upper = int(np.ceil(max(plot_summary["total_pages"].max(), plot_summary["detected_faces"].max() * 5) / 10000) * 10000)
ax_left.set_ylim(0, left_total_upper)
ax_left.yaxis.set_major_locator(MultipleLocator(10000))

ax_left_twin = ax_left.twinx()
left_line = ax_left_twin.plot(
    x,
    plot_summary["detected_faces"],
    color="#f58518",
    marker="o",
    linewidth=2.2,
    label="Detected faces",
)
ax_left_twin.set_ylabel("Detected faces")
ax_left_twin.set_ylim(0, left_total_upper / 5)
ax_left_twin.yaxis.set_major_locator(MultipleLocator(2000))

left_handles = [left_bar, left_line[0]]
ax_left.legend(left_handles, [handle.get_label() for handle in left_handles], loc="upper left")

ax_right = axes[1]
right_bar = ax_right.bar(x, plot_summary["pages_with_faces"], color="#72b7b2", alpha=0.9, label="Pages with faces")
ax_right.set_title("Pages with faces and normalized face presence")
ax_right.set_ylabel("Pages with faces")
ax_right.set_xticks(x, plot_summary["decade_label"], rotation=45, ha="right")
right_left_upper = int(np.ceil(max(plot_summary["pages_with_faces"].max(), plot_summary["pages_with_faces_per_100_pages"].max() * 400, plot_summary["detected_faces_per_100_pages"].max() * 400) / 2000) * 2000)
ax_right.set_ylim(0, right_left_upper)
ax_right.yaxis.set_major_locator(MultipleLocator(2000))

ax_right_twin = ax_right.twinx()
right_share_line = ax_right_twin.plot(
    x,
    plot_summary["pages_with_faces_per_100_pages"],
    color="#e45756",
    marker="o",
    linewidth=2.2,
    label="Pages with faces per 100 pages",
)
right_face_rate_line = ax_right_twin.plot(
    x,
    plot_summary["detected_faces_per_100_pages"],
    color="#54a24b",
    marker="s",
    linewidth=2.2,
    label="Detected faces per 100 pages",
)
ax_right_twin.set_ylabel("Rate per 100 pages")
ax_right_twin.set_ylim(0, right_left_upper / 400)
ax_right_twin.yaxis.set_major_locator(MultipleLocator(5))

right_handles = [right_bar, right_face_rate_line[0], right_share_line[0]]
ax_right.legend(right_handles, [handle.get_label() for handle in right_handles], loc="upper left")

fig.suptitle("Decade-level page totals and face presence", fontsize=15)
fig.text(0.5, 0.004, f"* {max_decade}s includes issues through {max_year} only.", ha="center", fontsize=10)
fig.savefig(summary_figure_svg, format="svg", bbox_inches="tight")
plt.show()

summary_figure_svg

In [ ]:
issue_face_share_by_decade = (
    decade_summary.loc[:, ["decade", "period_label", "issues_in_log", "issues_with_faces"]]
    .assign(issue_share_percent=lambda df: df["issues_with_faces"] / df["issues_in_log"] * 100)
    .sort_values("decade")
    .reset_index(drop=True)
)

forties_issue_counts = (
    download_log.loc[download_log["year"].between(1940, 1949), ["issue_id_short", "year"]]
    .assign(has_detected_face=lambda df: df["issue_id_short"].isin(set(faces["issue_id"])))
    .groupby("year", as_index=False)
    .agg(
        issues_in_year=("issue_id_short", "size"),
        issues_with_faces=("has_detected_face", "sum"),
    )
    .sort_values("year")
    .reset_index(drop=True)
)
forties_issue_counts["issues_with_faces"] = forties_issue_counts["issues_with_faces"].astype("int64")
forties_issue_counts["issue_share_percent"] = forties_issue_counts["issues_with_faces"] / forties_issue_counts["issues_in_year"] * 100

assert issue_face_share_by_decade["decade"].min() == 1900
assert issue_face_share_by_decade["issue_share_percent"].between(0, 100).all()
assert forties_issue_counts["year"].tolist() == list(range(1940, 1950))
assert forties_issue_counts["issue_share_percent"].between(0, 100).all()

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
fig.subplots_adjust(bottom=0.14, wspace=0.32)

ax_left = axes[0]
left_x = np.arange(len(issue_face_share_by_decade))
ax_left.bar(
    left_x,
    issue_face_share_by_decade["issue_share_percent"],
    color="#4c78a8",
    alpha=0.9,
)
ax_left.set_title("Issues with at least one detected face by decade", pad=14)
ax_left.set_xlabel("Period")
ax_left.set_ylabel("Issue share (%)")
ax_left.set_xticks(left_x, issue_face_share_by_decade["period_label"], rotation=45, ha="right")
ax_left.set_ylim(0, 108)
ax_left.yaxis.set_major_locator(MultipleLocator(10))

for xpos, value in zip(left_x, issue_face_share_by_decade["issue_share_percent"]):
    ax_left.text(xpos, min(value + 1.5, 104.5), f"{value:.1f}%", ha="center", va="bottom", fontsize=9)

ax_right = axes[1]
right_x = np.arange(len(forties_issue_counts))
bars = ax_right.bar(
    right_x,
    forties_issue_counts["issues_with_faces"],
    color="#72b7b2",
    alpha=0.9,
    label="Issues with at least one detected face",
)
ax_right.set_title("1940s yearly issue counts with detected faces")
ax_right.set_xlabel("Year")
ax_right.set_ylabel("Issues with detected faces")
ax_right.set_xticks(right_x, forties_issue_counts["year"].astype(str), rotation=45, ha="right")
ax_right.set_ylim(0, 50)
ax_right.yaxis.set_major_locator(MultipleLocator(10))

ax_right_twin = ax_right.twinx()
line = ax_right_twin.plot(
    right_x,
    forties_issue_counts["issue_share_percent"],
    color="#e45756",
    marker="o",
    linewidth=2.2,
    label="Issue share (%)",
)
ax_right_twin.set_ylabel("Issue share (%)")
ax_right_twin.set_ylim(0, 100)
ax_right_twin.yaxis.set_major_locator(MultipleLocator(20))

ax_right.legend([bars, line[0]], [bars.get_label(), line[0].get_label()], loc="upper left")

fig.suptitle("Issue-level face incidence from 1900 onward", fontsize=14)
fig.savefig(issue_incidence_figure_svg, format="svg", bbox_inches="tight")
plt.show()

issue_incidence_figure_svg

display(issue_face_share_by_decade[["period_label", "issues_in_log", "issues_with_faces", "issue_share_percent"]])
display(forties_issue_counts)


## Verification

The final cell reloads the written CSV, checks that the schema and row count match the in-memory summary, and prints the decades with the highest relative share of pages containing faces.

In [ ]:
reloaded_summary = pd.read_csv(summary_csv)

assert summary_figure_svg.exists(), f"Expected figure was not written: {summary_figure_svg}"
assert len(reloaded_summary) == len(decade_summary)
assert list(reloaded_summary.columns) == list(decade_summary.columns)

verification_columns = [
    "decade",
    "total_pages",
    "detected_faces",
    "pages_with_faces",
    "pages_with_faces_per_100_pages",
    "detected_faces_per_100_pages",
    "period_label",
]

reloaded_summary.sort_values("pages_with_faces_per_100_pages", ascending=False)[verification_columns].head(5)